# 🤖 학생 건강상태 분류 — 모델링 & 모델선택 (Lv3)

**EDA·파생변수 검증의 다음 단계** / 튜터 모드 · 발표용

이 노트북이 "증명"하는 것 (발표 장표에 그대로):
1. **변수 가지치기** — MI 검증 결과로 죽은 변수를 쳐내고 **정예 세트**만 사용 (근거 기록)
2. **범주형 인코딩 전략** — 순서형=서열(_ord), 명목형=OneHot (왜인지 설명)
3. **12개 모델 비교** — sklearn 7종 + **XGBoost·LightGBM·CatBoost** + 학습시간 측정
4. **시간 vs 성능 시각화** + **"왜 이 모델을 선택했는가" 의사결정 기록**
5. **선택 모델 튜닝 → 전체 학습 → submission.csv**
6. 표·그래프를 **PNG/CSV로 저장** (발표자료에 바로 삽입)

> 앞 노트북 안 돌려도 이 노트북 하나로 완결됩니다.


## 0. 라이브러리 설치 & import

XGBoost / LightGBM / CatBoost를 모두 설치해서 씁니다.


In [ ]:
!pip install -q lightgbm xgboost catboost
print('설치 완료')


In [ ]:
# (선택) 한글 폰트
import os
try:
    import matplotlib.font_manager as fm
    if not any('Nanum' in f.name for f in fm.fontManager.ttflist):
        os.system('apt-get -qq -y install fonts-nanum > /dev/null 2>&1')
        for fp in fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum']):
            fm.fontManager.addfont(fp)
except Exception as e:
    print('폰트 스킵:', e)


In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split, RandomizedSearchCV
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.base import clone

try:
    import matplotlib.font_manager as fm
    if any('Nanum' in f.name for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = 'NanumGothic'
except Exception: pass
plt.rcParams['axes.unicode_minus'] = False
RS = 42; np.random.seed(RS)
pd.set_option('display.max_columns', 60)
print('준비 완료 ✅')


## 1. 데이터 로드 + 파생변수 생성

파생변수 함수는 EDA 노트북과 동일합니다(모든 파생을 일단 만든 뒤, 다음 단계에서 정예만 선택).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/'
train = pd.read_csv(DATA_PATH + 'train.csv')
test  = pd.read_csv(DATA_PATH + 'test.csv')
sample_submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')
TARGET, ID = 'health_condition', 'id'
numeric_features = ['sleep_duration','heart_rate','bmi','calorie_expenditure','step_count','exercise_duration','water_intake']
categorical_features = ['diet_type','stress_level','sleep_quality','physical_activity_level','smoking_alcohol','gender']
feature_cols = numeric_features + categorical_features

HR_HI=train['heart_rate'].quantile(0.75); STEP_LO=train['step_count'].quantile(0.25); WATER_LO=train['water_intake'].quantile(0.25)
ord_maps={'stress_level':{'low':0,'medium':1,'high':2},'sleep_quality':{'poor':0,'average':1,'good':2},
 'physical_activity_level':{'sedentary':0,'moderate':1,'active':2},'smoking_alcohol':{'no':0,'occasional':1,'yes':2}}
def make_features(df):
    X=df.copy()
    for col,m in ord_maps.items(): X[col+'_ord']=X[col].map(m)
    X['sleep_debt']=(7-X['sleep_duration']).clip(lower=0); X['sleep_excess']=(X['sleep_duration']-9).clip(lower=0)
    X['sleep_ideal']=X['sleep_duration'].between(7,9).astype('float'); X.loc[X['sleep_duration'].isna(),'sleep_ideal']=np.nan
    X['bmi_cat']=pd.cut(X['bmi'],bins=[-np.inf,18.5,25,30,np.inf],labels=[0,1,2,3]).astype('float')
    X['bmi_abnormal']=(~X['bmi'].between(18.5,25)).astype('float'); X.loc[X['bmi'].isna(),'bmi_abnormal']=np.nan
    X['hr_high']=(X['heart_rate']>HR_HI).astype('float'); X.loc[X['heart_rate'].isna(),'hr_high']=np.nan
    X['steps_per_ex_min']=X['step_count']/(X['exercise_duration']+1); X['cal_per_step']=X['calorie_expenditure']/(X['step_count']+1)
    X['low_activity']=(X['step_count']<STEP_LO).astype('float'); X.loc[X['step_count'].isna(),'low_activity']=np.nan
    X['low_water']=(X['water_intake']<WATER_LO).astype('float'); X.loc[X['water_intake'].isna(),'low_water']=np.nan
    risk=pd.DataFrame(index=X.index)
    risk['r_sleep']=(X['sleep_duration']<6).astype(float); risk['r_stress']=(X['stress_level']=='high').astype(float)
    risk['r_sleepq']=(X['sleep_quality']=='poor').astype(float); risk['r_sed']=(X['physical_activity_level']=='sedentary').astype(float)
    risk['r_lowstep']=(X['step_count']<STEP_LO).astype(float); risk['r_smoke']=(X['smoking_alcohol']=='yes').astype(float)
    risk['r_bmi']=(~X['bmi'].between(18.5,25)).astype(float)
    X['lifestyle_risk_score']=risk.sum(axis=1); X['n_missing']=df[feature_cols].isnull().sum(axis=1)
    for c in feature_cols: X[c+'_isna']=df[c].isnull().astype('int8')
    return X
train_fe=make_features(train); test_fe=make_features(test)
print('train_fe', train_fe.shape)


## 2. ⭐ 변수 가지치기 (MI 검증 결과 반영)

앞 노트북 검증에서 배운 것:
- **상위 MI 파생변수**: `lifestyle_risk_score`, `*_ord`, `sleep_debt`, `sleep_ideal`, `bmi_cat`, `cal_per_step` 등 → **유지**
- **MI ≈ 0 (죽은 변수)**: 모든 `*_isna`, `n_missing`, `low_water`, `hr_high`, `sleep_excess`, `bmi_abnormal` → **제거** (차원만 늘리고 노이즈)

### 범주형 인코딩 전략 (질문 주신 부분 — 왜 이렇게?)
| 종류 | 변수 | 인코딩 | 이유 |
|---|---|---|---|
| **순서형(ordinal)** | stress_level, sleep_quality, physical_activity_level, smoking_alcohol | **서열 `_ord` 숫자 하나** | 'low<medium<high' 처럼 **순서 자체가 정보**. 숫자로 주면 트리가 'stress≥2면 위험' 분기를 바로 만들고, OneHot보다 **차원↓**. 원본 OneHot은 순서를 버려서 중복 → **제거** |
| **명목형(nominal)** | diet_type, gender | **OneHot** | 순서가 없음(veg vs non-veg에 대소 없음). 숫자를 매기면 **가짜 순서**가 생겨 왜곡 → OneHot이 정답 |

> 핵심: **순서가 있으면 서열(숫자), 없으면 OneHot.** 순서형에 OneHot과 _ord를 둘 다 넣는 건 같은 정보를 두 번 주는 낭비라 정리합니다.


In [ ]:
# --- 정예 변수 세트 ---
num_final = numeric_features + [
    'lifestyle_risk_score','sleep_debt','sleep_ideal','bmi_cat',
    'cal_per_step','steps_per_ex_min','low_activity',
    'stress_level_ord','sleep_quality_ord','physical_activity_level_ord','smoking_alcohol_ord',
]
cat_final = ['diet_type','gender']   # 명목형만 OneHot

dropped = ['sleep_excess','bmi_abnormal','hr_high','low_water','n_missing'] + \
          [c+'_isna' for c in feature_cols] + \
          ['stress_level','sleep_quality','physical_activity_level','smoking_alcohol']  # 순서형 원본은 _ord로 대체
print(f'유지: 수치/서열 {len(num_final)}개 + 명목 OneHot {len(cat_final)}개')
print(f'제거: {len(dropped)}개 (죽은 변수 + 순서형 원본 중복)')
print('\n제거 목록:', dropped)


## 3. 전처리 아키텍처 (Pipeline)

- 수치/서열 → 중앙값 대치 + 표준화
- 명목형 → 최빈값 대치 + OneHot
- 이 `preprocessor`를 12개 모델 앞에 동일하게 붙임 = "머신러닝 아키텍처" 요건


In [ ]:
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                             ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', numeric_pipe, num_final), ('cat', categorical_pipe, cat_final)])
print('preprocessor 구성 완료 ✅ | 입력 컬럼:', len(num_final)+len(cat_final))


## 4. 모델 12종 + "왜 이 모델을 넣었는가"

발표에서 "아무거나 10개"가 아니라 **의도를 갖고 계열별로 배치**했다고 말할 수 있게 정리했습니다.

| # | 모델 | 계열 | 넣은 이유 / 기대 |
|---|---|---|---|
| 1 | LogisticRegression | 선형 | 해석 쉬운 기준선. 파생변수 효과가 선형모델에서 큰지 확인 |
| 2 | RidgeClassifier | 선형+정규화 | 다중공선성에 강한 선형 |
| 3 | LinearSVC | 선형(마진) | 마진 기반 관점의 선형 |
| 4 | KNN | 거리기반 | 국소 패턴 비선형 기준선 |
| 5 | GaussianNB | 확률기반 | 초고속 확률 기준선 |
| 6 | DecisionTree | 트리 | 앙상블 전 단일 트리 기준 |
| 7 | RandomForest | 배깅 | 분산↓, 안정적 앙상블 |
| 8 | ExtraTrees | 배깅(더 랜덤) | RF 변형 비교 |
| 9 | HistGradientBoosting | 부스팅(sklearn) | 빠르고 강한 부스팅 |
| 10 | **LightGBM** | 부스팅(leaf-wise) | 대용량에 빠름, 캐글 강자 |
| 11 | **XGBoost** | 부스팅(level-wise) | 캐글 표준, 규제 강함 |
| 12 | **CatBoost** | 부스팅(범주형 특화) | 순서형/범주형에 강함 |

> **불균형 대응**: 지원 모델은 `class_weight='balanced'`(CatBoost는 `auto_class_weights`, XGBoost는 아래 래퍼로 sample_weight 자동 적용). KNN/GaussianNB는 미지원 → 기준선 역할.


In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RS),
    'RidgeClassifier':    RidgeClassifier(class_weight='balanced', random_state=RS),
    'LinearSVC':          LinearSVC(class_weight='balanced', random_state=RS),
    'KNN':                KNeighborsClassifier(n_neighbors=25, n_jobs=-1),
    'GaussianNB':         GaussianNB(),
    'DecisionTree':       DecisionTreeClassifier(class_weight='balanced', max_depth=12, random_state=RS),
    'RandomForest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=RS),
    'ExtraTrees':         ExtraTreesClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=RS),
    'HistGradientBoosting': HistGradientBoostingClassifier(class_weight='balanced', max_iter=300, random_state=RS),
}
# --- 부스팅 3종 (설치 실패 시 자동 제외) ---
try:
    from lightgbm import LGBMClassifier
    models['LightGBM'] = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63,
                                        class_weight='balanced', random_state=RS, n_jobs=-1, verbose=-1)
except Exception as e: print('LightGBM 제외:', e)
try:
    from xgboost import XGBClassifier
    from sklearn.utils.class_weight import compute_sample_weight
    class BalancedXGB(XGBClassifier):           # 불균형 대응: fit 때 sample_weight 자동 주입
        def fit(self, X, y, **kw):
            return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)
    models['XGBoost'] = BalancedXGB(n_estimators=300, learning_rate=0.1, max_depth=6, tree_method='hist',
                                    random_state=RS, n_jobs=-1, verbosity=0, eval_metric='mlogloss')
except Exception as e: print('XGBoost 제외:', e)
try:
    from catboost import CatBoostClassifier
    models['CatBoost'] = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1,
                                            auto_class_weights='Balanced', random_state=RS, verbose=0)
except Exception as e: print('CatBoost 제외:', e)

print(f'\n총 {len(models)}개 모델:', list(models.keys()))


## 5. 비교용 층화 샘플 (10만 건)

전체 69만으로 12개를 돌리면 오래 걸립니다. **10만 건 층화 샘플**로 상대 비교하고, 최종 모델만 전체로 재학습.
타깃은 XGBoost 호환을 위해 **정수 라벨(LabelEncoder)** 로 변환합니다(모든 지표는 동일하게 계산됨).


In [ ]:
COMPARE_SAMPLE = 100000
samp = train_fe.groupby(TARGET, group_keys=False).apply(
    lambda s: s.sample(min(len(s), int(COMPARE_SAMPLE*len(s)/len(train_fe))+1), random_state=RS))
X_samp = samp[num_final + cat_final]
le = LabelEncoder().fit(train_fe[TARGET])         # fit / at-risk / unhealthy -> 0/1/2
y_samp = le.transform(samp[TARGET])
print('비교 샘플:', X_samp.shape, '| 클래스:', dict(zip(le.classes_, np.bincount(y_samp))))


## 6. ⭐ 12개 모델 비교표 (학습시간 측정)

- `cross_validate`가 fit_time 반환 → **시간 측정 요건**. 깨끗한 시간 측정을 위해 CV는 순차(n_jobs=1) 실행.
- 지표: **balanced_accuracy**(주지표) + f1_macro, 3-fold.
- ⏳ 부스팅 3종 때문에 수 분~십수 분 걸릴 수 있어요.


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RS)
scoring = ['balanced_accuracy', 'f1_macro']
rows = []
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', model)])
    try:
        r = cross_validate(pipe, X_samp, y_samp, cv=cv, scoring=scoring, n_jobs=1, error_score='raise')
        rows.append({'model':name,
                     'balanced_acc':r['test_balanced_accuracy'].mean(),
                     'std':r['test_balanced_accuracy'].std(),
                     'f1_macro':r['test_f1_macro'].mean(),
                     'fit_time(s)':r['fit_time'].mean()})
        print(f'✔ {name:22s} bal_acc={rows[-1]["balanced_acc"]:.4f}  ±{rows[-1]["std"]:.4f}  time={rows[-1]["fit_time(s)"]:.1f}s')
    except Exception as e:
        print(f'✘ {name:22s} 실패: {str(e)[:70]}')

result = pd.DataFrame(rows).sort_values('balanced_acc', ascending=False).reset_index(drop=True)
display(result.round(4))

# 발표용 저장
import os
RESULT_DIR = DATA_PATH + 'results/'; os.makedirs(RESULT_DIR, exist_ok=True)
result.round(4).to_csv(RESULT_DIR + 'model_comparison.csv', index=False)
print('\n표 저장:', RESULT_DIR + 'model_comparison.csv')


## 7. 시간 vs 성능 산점도 (요건)

**왼쪽 위 = 빠르면서 정확(가성비 최고)**, 오른쪽 위 = 느리지만 정확.


In [ ]:
plt.figure(figsize=(11,6.5))
sns.scatterplot(data=result, x='fit_time(s)', y='balanced_acc', s=140, hue='model', legend=False)
for _, r in result.iterrows():
    plt.annotate(r['model'], (r['fit_time(s)'], r['balanced_acc']), xytext=(6,4),
                 textcoords='offset points', fontsize=9)
plt.xlabel('학습 시간 (초, 낮을수록 좋음)'); plt.ylabel('balanced accuracy (높을수록 좋음)')
plt.title('모델별 시간 대비 성능')
plt.tight_layout()
plt.savefig(RESULT_DIR + 'model_comparison_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장:', RESULT_DIR + 'model_comparison_scatter.png')


## 8. ⭐ "왜 이 모델을 선택했는가" — 의사결정 기록

발표의 핵심 논리입니다. 아무거나 1등을 고르는 게 아니라 **기준**을 정하고 고릅니다.

### 선택 기준 (우선순위)
1. **balanced_acc (주지표)** — 불균형 다중분류의 핵심
2. **안정성(std)** — fold별 편차가 작을수록 신뢰
3. **효율(fit_time)** — 비슷한 성능이면 빠른 쪽 (재학습·튜닝·서비스화에 유리)
4. **과적합 여지** — 지나치게 복잡한 모델은 경계

아래 셀이 기준에 따라 후보를 자동 정리해 줍니다. 최종 문장은 실제 숫자를 보고 채우세요.


In [ ]:
top = result.head(3).copy()
best = result.iloc[0]
# 성능 상위권(1등과 0.003 이내) 중 가장 빠른 = '가성비 선택' 후보
near = result[result['balanced_acc'] >= best['balanced_acc'] - 0.003]
value_pick = near.sort_values('fit_time(s)').iloc[0]

print('■ 성능 1위      :', best['model'], f"(bal_acc={best['balanced_acc']:.4f}, {best['fit_time(s)']:.1f}s)")
print('■ 가성비 후보    :', value_pick['model'], f"(bal_acc={value_pick['balanced_acc']:.4f}, {value_pick['fit_time(s)']:.1f}s)")
print('■ 상위 3개:')
display(top.round(4))

print('\n[발표 문장 템플릿]')
print(f"→ 성능만 보면 '{best['model']}'가 1위지만, 성능이 사실상 동급(±0.003)이면서")
print(f"  학습이 더 빠른 '{value_pick['model']}'를 최종 후보로 본다. (성능·안정성·효율 종합)")
print("  실제 숫자를 보고 팀과 합의해 최종 1개를 아래 FINAL_MODEL 에 지정하세요.")


In [ ]:
# 최종 모델 지정 (위 결과 보고 이름만 바꾸면 됨. 기본=성능 1위)
FINAL_MODEL = result.iloc[0]['model']
print('최종 선택 모델:', FINAL_MODEL)


## 9. 선택 모델 튜닝 (RandomizedSearchCV)

선택 모델을 샘플에서 가볍게 튜닝합니다(n_iter=15, 3-fold). 부스팅 계열이면 대표 하이퍼파라미터를 탐색.


In [ ]:
param_grids = {
    'LightGBM': {'clf__n_estimators':[300,500,800],'clf__learning_rate':[0.03,0.05,0.1],
                 'clf__num_leaves':[31,63,127],'clf__subsample':[0.8,1.0],'clf__colsample_bytree':[0.7,0.9,1.0]},
    'XGBoost':  {'clf__n_estimators':[300,500,800],'clf__learning_rate':[0.03,0.05,0.1],
                 'clf__max_depth':[4,6,8],'clf__subsample':[0.8,1.0],'clf__colsample_bytree':[0.7,0.9,1.0]},
    'CatBoost': {'clf__iterations':[300,500,800],'clf__learning_rate':[0.03,0.05,0.1],'clf__depth':[4,6,8]},
    'HistGradientBoosting': {'clf__max_iter':[300,500],'clf__learning_rate':[0.03,0.05,0.1],
                             'clf__max_leaf_nodes':[31,63,127],'clf__l2_regularization':[0,1.0,5.0]},
    'RandomForest': {'clf__n_estimators':[200,400],'clf__max_depth':[None,16,24],'clf__min_samples_leaf':[1,3,5]},
    'ExtraTrees':   {'clf__n_estimators':[200,400],'clf__max_depth':[None,16,24],'clf__min_samples_leaf':[1,3,5]},
}
base_pipe = Pipeline([('prep', preprocessor), ('clf', clone(models[FINAL_MODEL]))])
if FINAL_MODEL in param_grids:
    search = RandomizedSearchCV(base_pipe, param_grids[FINAL_MODEL], n_iter=15, cv=cv,
                                scoring='balanced_accuracy', n_jobs=1, random_state=RS, verbose=1)
    t0=time.time(); search.fit(X_samp, y_samp)
    print(f'\n튜닝 완료 {time.time()-t0:.1f}s | best CV bal_acc={search.best_score_:.4f}')
    print('best params:', {k.replace("clf__",""):v for k,v in search.best_params_.items()})
    best_pipe = search.best_estimator_
else:
    print(f'{FINAL_MODEL}: 별도 그리드 없음 → 기본 파라미터로 진행')
    best_pipe = base_pipe.fit(X_samp, y_samp)


## 10. 최종 검증 → 전체 학습 → 제출

1. 전체 train 80/20 층화 분할 → hold-out으로 정직한 성능 확인 (혼동행렬 저장)
2. 전체 train 재학습 → test 예측 → `submission.csv` (라벨 문자열로 복원)


In [ ]:
X_all = train_fe[num_final + cat_final]; y_all = le.transform(train_fe[TARGET])
X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=RS)

best_pipe.fit(X_tr, y_tr)
pv = best_pipe.predict(X_val)
print('Validation balanced_accuracy:', round(balanced_accuracy_score(y_val, pv), 4))
print('Validation f1_macro        :', round(f1_score(y_val, pv, average='macro'), 4))
print('\n', classification_report(y_val, pv, target_names=le.classes_))

cm = confusion_matrix(y_val, pv)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('예측'); plt.ylabel('실제'); plt.title(f'Confusion Matrix — {FINAL_MODEL}')
plt.tight_layout(); plt.savefig(RESULT_DIR + 'confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# 전체 데이터 재학습 → 제출
t0=time.time(); best_pipe.fit(X_all, y_all); print(f'전체 재학습 {time.time()-t0:.1f}s')
test_pred = le.inverse_transform(best_pipe.predict(test_fe[num_final + cat_final]))
submission = pd.DataFrame({ID: test[ID], TARGET: test_pred})
submission.to_csv(DATA_PATH + 'submission_final.csv', index=False)
print('저장:', DATA_PATH + 'submission_final.csv', submission.shape)
print(submission[TARGET].value_counts()); submission.head()


## 11. 변수 중요도 (선택 모델)

정예 변수 중 무엇이 실제로 결정에 쓰였는지. 파생변수(`lifestyle_risk_score`, `*_ord` 등)가 상위면 설계가 통한 것.


In [ ]:
clf = best_pipe.named_steps['clf']
feat_names = best_pipe.named_steps['prep'].get_feature_names_out()
if hasattr(clf, 'feature_importances_'):
    imp = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False).head(20)
    title = f'{FINAL_MODEL} 변수 중요도 (상위 20)'
else:
    from sklearn.inspection import permutation_importance
    idx = np.random.RandomState(RS).choice(len(X_val), size=min(5000,len(X_val)), replace=False)
    pi = permutation_importance(best_pipe, X_val.iloc[idx], y_val[idx], scoring='balanced_accuracy',
                                n_repeats=5, random_state=RS, n_jobs=-1)
    imp = pd.Series(pi.importances_mean, index=(num_final+cat_final)).sort_values(ascending=False).head(20)
    title = f'{FINAL_MODEL} Permutation 중요도 (상위 20)'
plt.figure(figsize=(9,7)); sns.barplot(x=imp.values, y=imp.index, palette='mako')
plt.title(title); plt.tight_layout()
plt.savefig(RESULT_DIR + 'feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()
display(imp.round(4).to_frame('importance'))


## 12. 발표용 정리 & SCQA

### 저장된 발표 자료 (results/ 폴더)
- `model_comparison.csv` — 12모델 비교표
- `model_comparison_scatter.png` — 시간 대비 성능
- `confusion_matrix.png` — 최종 모델 혼동행렬
- `feature_importance.png` — 변수 중요도

### 모델선택 스토리라인 (장표 흐름 제안)
1. 문제: 불균형 3분류 → **balanced_accuracy**로 평가
2. 12개 모델을 계열별 의도를 갖고 비교 (선형/거리/트리/부스팅)
3. 시간 대비 성능으로 **후보 압축** → 기준(성능·안정성·효율)으로 **최종 1개 선택**
4. 튜닝 → 전체 학습 → 제출 점수

### SCQA (서비스 기획)
- **S**: 학생 생활습관 데이터 수집 가능
- **C**: 건강 이상은 증상 후 발견 → 개입이 늦고, 상담 인력은 한정
- **Q**: 생활습관만으로 위험군(at-risk/unhealthy)을 미리 선별 가능한가?
- **A**: 본 모델로 **건강 위험군 조기경보 → 맞춤 코칭 추천** 서비스
- **액션플랜**: 설문/웨어러블 입력 → 모델 위험도 출력 → 대시보드 + 상위 위험요인(중요도 기반) → 위험군 코칭 알림/상담 연계
- **한계·향후**: 합성 라벨 가능성(임상검증 필요), 결측 많음(수집품질 개선), 시계열 반영 시 조기경보↑

> 이 노트북 실행 후 나온 표·그래프를 주시면 **발표 PPT 초안**으로 바로 만들어 드릴 수 있어요.
